# dcgan-wrapper-netG-netD — worked example 2: Build two optimizers from one wrapper's subnets

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `dcgan-wrapper-netG-netD`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Concept

Because the wrapper exposes `netG` and `netD` as distinct submodules, you can build a **separate** optimizer per subnet by passing `wrapper.netG.parameters()` and `wrapper.netD.parameters()`. This is exactly what a GAN training loop needs: the generator and discriminator are optimized in alternating, independent steps, so they must not share an optimizer or a parameter list.

## Worked solution

**Step 1 — build the wrapper.** Same idiom: `nn.Module` subclass, `super().__init__()` first, then assign `netG` and `netD`. No `forward`.

**Step 2 — pull each subnet's parameter iterator.** `wrapper.netG.parameters()` yields only the generator's params; `wrapper.netD.parameters()` only the discriminator's. Because they were registered as separate submodules, these two sets are disjoint.

**Step 3 — construct two optimizers.** `optG = t.optim.Adam(wrapper.netG.parameters(), lr=2e-4, betas=(0.5, 0.999))` and similarly `optD`. The DCGAN paper's hyperparameters (`lr=2e-4`, `betas=(0.5, 0.999)`) are the conventional choice.

**Step 4 — verify disjointness.** We collect the `id()` of every parameter tensor each optimizer manages and confirm the two sets do not overlap, and that together they cover every parameter in the wrapper. This guarantees a `optG.step()` never nudges discriminator weights and vice versa.

In [ ]:
from torch import nn
import torch as t

def build_dcgan_optimizers(generator, discriminator):
    class DCGAN(nn.Module):
        def __init__(self, netG, netD):
            super().__init__()
            self.netG = netG
            self.netD = netD
    wrapper = DCGAN(generator, discriminator)
    optG = t.optim.Adam(wrapper.netG.parameters(), lr=2e-4, betas=(0.5, 0.999))
    optD = t.optim.Adam(wrapper.netD.parameters(), lr=2e-4, betas=(0.5, 0.999))
    return wrapper, optG, optD

t.manual_seed(0)
gen = nn.Sequential(nn.Linear(10, 32), nn.ReLU(), nn.Linear(32, 64))
disc = nn.Sequential(nn.Linear(64, 16), nn.ReLU(), nn.Linear(16, 1))
wrapper, optG, optD = build_dcgan_optimizers(gen, disc)

g_ids = {id(p) for grp in optG.param_groups for p in grp['params']}
d_ids = {id(p) for grp in optD.param_groups for p in grp['params']}
all_ids = {id(p) for p in wrapper.parameters()}
print('disjoint:', g_ids.isdisjoint(d_ids), '| cover all:', (g_ids | d_ids) == all_ids, '| nG:', len(g_ids), '| nD:', len(d_ids))